# Busqueda No Informada

## Instalar librerias
```python
%pip install pandas
```

## Importar librerias

```python

from collections import deque #pila y cola
import random
import heapq #heap
from IPython.display import clear_output
import time #medir tiempo
import pandas as pd
```

## Nodo

# El nodo es la estructura de datos que se utiliza para representar el estado del problema.

```python
class Nodo:
    def __init__(self, dato, padre=None, distancia=0):
        self.dato = dato
        self.padre = padre
        self.H = distancia
        if padre == None:
            self.profundidad = 0
        else:
            self.profundidad = padre.profundidad + 1

    def GenerarSucesores(self):
        return self.dato.GenerarSucesores()

    def __eq__(self, __o) -> bool:
        if isinstance(__o, Nodo):
            return self.dato == __o.dato
        if isinstance(__o, type(self.dato)):
            return self.dato == __o
        return False

    # < override
    def __lt__(self, __o) -> bool:
        if isinstance(__o, Nodo):
            return self.Heuristica() < __o.Heuristica()
        return False

    # > override
    def __gt__(self, __o) -> bool:
        if isinstance(__o, Nodo):
            return self.Heuristica() > __o.Heuristica()
        return False

    def Heuristica(self):
        return self.H + self.profundidad

    def __hash__(self) -> int:
        hsh = str(self.dato)

        return hash(hsh)

    def __str__(self):
        return str(self.dato.__str__())

```

## Busqueda Primero en Anchura

```python
def BusquedaEnAnchura(estado_inicial,estado_final):
    totalnodos=1
    nodoactual=Nodo(estado_inicial,None)
    nodosgenerado=deque()
    nodosvisitados=set()
    inicio = time.perf_counter()

    # busqueda

    while nodoactual.dato !=estado_final:

        sucesores=nodoactual.GenerarSucesores()
        totalnodos+=len(sucesores)

        for sucesor in sucesores:
            temp= Nodo(sucesor,nodoactual)
            if temp not in nodosvisitados:
                nodosgenerado.append(temp)

        nodosvisitados.add(nodoactual)

        while nodoactual in nodosvisitados:
            nodoactual=nodosgenerado.popleft()

    fin=time.perf_counter()
    camino=[]
    while nodoactual:
        camino.append(nodoactual.dato)
        nodoactual=nodoactual.padre
    camino.reverse()
    return camino,totalnodos,fin-inicio

```

## Busqueda Primero en Profundidad

```python
def BusquedaEnProfundidad(estado_inicial,estado_final):
    totalnodos=1
    nodoactual=Nodo(estado_inicial,None)
    nodosgenerado=deque()
    nodosvisitados=set()
    inicio=time.perf_counter()

#busqueda
    while nodoactual.dato !=estado_final:
        sucesores=nodoactual.GenerarSucesores()
        totalnodos+=len(sucesores)
        for sucesor in sucesores:
            temp= Nodo(sucesor,nodoactual)
            if temp not in nodosvisitados:
                nodosgenerado.append(temp)

        nodosvisitados.add(nodoactual)
        while nodoactual in nodosvisitados:
            nodoactual=nodosgenerado.pop()
    camino=[]
    while nodoactual:
        camino.append(nodoactual.dato)
        nodoactual=nodoactual.padre
    camino.reverse()
    fin=time.perf_counter()
    return camino ,totalnodos,fin-inicio

```

## Busqueda Bidireccional

```python

def BusquedaBidireccional(estado_inicial,estado_final):
    totalnodos = 2
    respuesta = False

    nodoinicial = Nodo(estado_inicial, None)
    nodofinal = Nodo(estado_final, None)

    nodosgeneradoinicial = deque([nodoinicial])
    nodosgeneradofinal = deque([nodofinal])

    nodosvisitadosinicial = {nodoinicial}
    nodosvisitadosfinal = {nodofinal}

    inicio = time.perf_counter()

    while not respuesta and nodosgeneradoinicial and nodosgeneradofinal:
        # Procesar sucesores desde el lado inicial
        nodoinicial = nodosgeneradoinicial.popleft()
        sucesoresinicial = nodoinicial.GenerarSucesores()
        totalnodos += len(sucesoresinicial)

        for sucesor in sucesoresinicial:
            nodo = Nodo(sucesor, nodoinicial)
            if nodo in nodosvisitadosinicial:
                continue
            if nodo in nodosgeneradofinal:
                respuesta = True
                nodofinal = nodosgeneradofinal[nodosgeneradofinal.index(nodo)]
                break
            nodosgeneradoinicial.append(nodo)
            nodosvisitadosinicial.add(nodo)

        if respuesta:
            break

        # Procesar sucesores desde el lado final
        nodofinal = nodosgeneradofinal.popleft()
        sucesoresfinal = nodofinal.GenerarSucesores()
        totalnodos += len(sucesoresfinal)

        for sucesor in sucesoresfinal:
            nodo = Nodo(sucesor, nodofinal)
            if nodo in nodosvisitadosfinal:
                continue
            if nodo in nodosgeneradoinicial:
                respuesta = True
                nodoinicial = nodosgeneradoinicial[nodosgeneradoinicial.index(nodo)]
                break
            nodosgeneradofinal.append(nodo)
            nodosvisitadosfinal.add(nodo)

    fin = time.perf_counter()
    camino=[]

    while nodoinicial:
        camino.append(nodoinicial.dato)
        nodoinicial=nodoinicial.padre
    camino.reverse()

    while nodofinal:
        camino.append(nodofinal.dato)
        nodofinal=nodofinal.padre

    return camino,totalnodos,fin-inicio

```

## Tablero de los Caballos

```python
class Tablero:

    def __init__(self,configuracion):
        self.configuracion=configuracion

    def __eq__(self, __o: object) -> bool:
        if isinstance(__o, Tablero):
            for i in range(9):
                if self.configuracion[i]!=__o.configuracion[i]:
                    return False
            return True
        if isinstance(__o, list):
            for i in range(9):
                if self.configuracion[i]!=__o[i]:
                    return False
            return True
        return False

    def __hash__(self) -> int:
        return hash(str(self.configuracion))

    def __str__(self):
        return str(f"|{self.configuracion[0]}|{self.configuracion[1]}|{self.configuracion[2]}|\n\
|{self.configuracion[3]}|{self.configuracion[4]}|{self.configuracion[5]}|\n\
|{self.configuracion[6]}|{self.configuracion[7]}|{self.configuracion[8]}|\n\n")

    def GenerarSucesores(self):
        sucesores=[]
        movimientos_validos = {
        0: [5, 7],
        1: [6, 8],
        2: [3, 7],
        3: [2, 8],
        5: [0, 6],
        6: [1, 5],
        7: [0, 2],
        8: [1, 3]
    }
        for i in range(9):
            if self.configuracion[i] != 0:
                for j in movimientos_validos[i]:
                    if self.configuracion[j] == 0:
                        nueva_configuracion=self.configuracion.copy()
                        nueva_configuracion[j]=self.configuracion[i]
                        nueva_configuracion[i]=0
                        sucesores.append(Tablero(nueva_configuracion))
        return sucesores
```

## Ejecución de los algoritmos

```python
estado_inicial=Tablero(
    [1,0,1,
    0,0,0,
    2,0,2])
estado_final=Tablero(
    [2,0,2,
    0,0,0,
    1,0,1])


resultados = []

algoritmos = {
    "BusquedaEnAnchura": BusquedaEnAnchura,
    "BusquedaEnProfundidad": BusquedaEnProfundidad,
    "BusquedaBidireccional": BusquedaBidireccional,
}

for nombre, algoritmo in algoritmos.items():
    resultado, totalnodos, tiempo = algoritmo(estado_inicial, estado_final)
    pasos = len(resultado) - 1
    tiempo = tiempo * 1000  # Convertir a milisegundos
    resultados.append(
        {
            "Algoritmo": nombre,
            "Total de pasos": pasos,
            "Total de nodos": totalnodos,
            "Duración (ms)": tiempo,
            "Camino": resultado,
        }
    )

df = pd.DataFrame(resultados)
df

# ver el camino
# indice = 0
# for i in range(len(df["Camino"][indice])):
#     clear_output(wait=True)
#     print(df["Camino"][indice][i])
#     time.sleep(1)

```

# Busqueda Informada

## Algoritmo A*

```python
def Astar(estado_inicial,estado_final):
    totalnodos=1
    nodoactual=Nodo(estado_inicial,None,estado_inicial.Costo(estado_final))
    nodosgenerado=[]

    nodosvisitados=set()
    heapq.heapify(nodosgenerado)

    inicio=time.perf_counter()

    while nodoactual.dato !=estado_final:
        sucesores=nodoactual.GenerarSucesores()

        totalnodos+=len(sucesores)

        for sucesor in sucesores:
            temp= Nodo(sucesor,nodoactual,sucesor.Costo(estado_final))

            if temp not in nodosvisitados:
                heapq.heappush(nodosgenerado, temp)
        nodosvisitados.add(nodoactual)

        while nodoactual in nodosvisitados:
            nodoactual= heapq.heappop(nodosgenerado)
    camino=[]
    while nodoactual:
        camino.append(nodoactual.dato)
        nodoactual=nodoactual.padre
    camino.reverse()
    fin=time.perf_counter()
    return camino,totalnodos,fin-inicio
```

## Busqueda de ruta en un mapa



# Generar mapa

```python
def generar_mundo(n):
    mapa = [[0 for _ in range(n)] for _ in range(n)]
    mapa[0][0] = "S"
    destino=[ random.randint(1,n-1), random.randint(1,n-1)]
    mapa[destino[0]][destino[1]]   = "E"
    for i in range(n):
        for j in range(n):
            if i == 0 and j == 0:
                continue
            if i ==destino[0] and j == destino[1]:
                continue
            if random.random() < 0.1:
                mapa[i][j] = 1
    return mapa,destino

```

# clase mapa

```python
class Mapa:
    def __init__(self,configuracion,cordenadas) -> None:
        self.configuracion=configuracion
        self.cordenadas=cordenadas
        self.tamano=len(configuracion)

    def GenerarSucesores(self):
            sucesores=[]
            movimientos_validos = [[0, 1], [1, 0], [0, -1], [-1, 0]]

            for movimiento in movimientos_validos:
                x=self.cordenadas[0]+movimiento[0]
                y=self.cordenadas[1]+movimiento[1]

                if x>=0 and x<self.tamano and y>=0 and y<self.tamano:
                    if self.configuracion[x][y]!=1:
                        nueva_configuracion=self.configuracion.copy()

                        sucesores.append(Mapa(nueva_configuracion,[x,y]))

            return sucesores

    def __eq__(self, __o: object) -> bool:
        if isinstance(__o, Mapa):
            return self.cordenadas==__o.cordenadas
        return self.cordenadas==__o

    def __hash__(self) -> int:
        return hash(str(self.configuracion))

    def __str__(self):
        mapa_string = ""

        for i, fila in enumerate(self.configuracion):
            for j, elemento in enumerate(fila):
                if i == self.cordenadas[0] and j == self.cordenadas[1] and elemento != "E" and elemento != "S":

                    mapa_string += "* "
                else:
                    mapa_string += str(elemento) + " "
            mapa_string += "\n"
        return mapa_string

    def Costo(self,estado_final):
        return abs(self.cordenadas[0]-estado_final.cordenadas[0])+abs(self.cordenadas[1]-estado_final.cordenadas[1])
```

## Crear problema

```python
inicio=[0,0]
configuracion,destino=generar_mundo(10)

inicio=Mapa(configuracion.copy(),inicio)
destino=Mapa(configuracion.copy(),destino)
```

## Ejecución de los algoritmos

```python
# Lista para almacenar los resultados
resultados = []

algoritmos = {
    "BusquedaEnAnchura": BusquedaEnAnchura,
    "BusquedaEnProfundidad": BusquedaEnProfundidad,
    "BusquedaBidireccional": BusquedaBidireccional,
    "Astar": Astar,
}

for nombre, algoritmo in algoritmos.items():
    resultado, totalnodos, tiempo = algoritmo(inicio, destino)
    pasos = len(resultado) - 1
    tiempo = tiempo * 1000
    resultados.append(
        {
            "Algoritmo": nombre,
            "Total de pasos": pasos,
            "Total de nodos": totalnodos,
            "Duración (ms)": tiempo,
            "Camino": resultado,
        }
    )

df = pd.DataFrame(resultados)
df

# ver el camino
#indice=0
# for i in range(len(df["Camino"][indice])):
#     clear_output(wait=True)
#     print(df["Camino"][indice][i])):
#     time.sleep(1)

```

# npuzzle
 

implementar posibilidad n y medir si es resoluble 

### clase 8 puzzle

```python
class N8puzzle:
    def __init__(self, configuracion):
        self.configuracion = configuracion

    def GenerarSucesores(self ):
        sucesores = []
        movimientos_validos = {
                    0: [1, 3],
                    1: [0, 2, 4],
                    2: [1, 5],
                    3: [0, 4, 6],
                    4: [1, 3, 5, 7],
                    5: [2, 4, 8],
                    6: [3, 7],
                    7: [4, 6, 8],
                    8: [5, 7]
                }

        i = self.configuracion.index(0)
        for movimiento in movimientos_validos[i]:
            nueva_configuracion =  self.configuracion.copy()
            nueva_configuracion[i]=nueva_configuracion[movimiento]
            nueva_configuracion[movimiento] =0
            sucesores.append(N8puzzle(nueva_configuracion))
        return sucesores

    def __eq__(self, __o: object) -> bool:
        if isinstance(__o, N8puzzle):
            return self.configuracion == __o.configuracion
        if isinstance(__o, list):
            return self.configuracion == __o
        return False

    def __hash__(self) -> int:
        return hash(str(self.configuracion))

    def __str__(self):
        return str(f"|{self.configuracion[0]}|{self.configuracion[1]}|{self.configuracion[2]}|\n\
|{self.configuracion[3]}|{self.configuracion[4]}|{self.configuracion[5]}|\n\
|{self.configuracion[6]}|{self.configuracion[7]}|{self.configuracion[8]}|\n\n")

    def Costo(self, estado_final):
        posicion_valor_estado1 = {}
        for i, valor in enumerate(self.configuracion):
            posicion_valor_estado1[valor] = i

        distancia = 0
        for i, valor in enumerate(estado_final.configuracion):
            if valor == 0:
                continue
            # Obtener la posición en estado1 del mismo valor en estado2
            posicion_estado1 = posicion_valor_estado1[valor]
            # Calcular la distancia de Manhattan y sumarla
            distancia += abs(i // 3 - posicion_estado1 // 3) + abs(i % 3 - posicion_estado1 % 3)

        return distancia

```

## problemas

```python

problema1=N8puzzle([8,7,5,
	                3,0,1,
	                4,2,6])

problema2=N8puzzle([5,4,0,
                    6,1,8,
                    7,3,2])
problema3=N8puzzle([8, 6, 7,
                    2, 5, 4,
                    3, 0, 1])
problema4=N8puzzle([6, 4, 7,
	                8, 5, 0,
	                3, 2, 1])

solucion=N8puzzle([1,2,3,
                4,5,6,
                7,8,0])
```

## Ejecución de los algoritmos

```python
resultados = []

algoritmos = {
    "BusquedaEnAnchura": BusquedaEnAnchura,
    "BusquedaEnProfundidad": BusquedaEnProfundidad,
    "BusquedaBidireccional": BusquedaBidireccional,
    "Astar": Astar,
}

for nombre, algoritmo in algoritmos.items():
    resultado, totalnodos, tiempo = algoritmo(problema1, solucion)
    pasos = len(resultado) - 1
    tiempo = tiempo * 1000
    resultados.append(
        {
            "Algoritmo": nombre,
            "Total de pasos": pasos,
            "Total de nodos": totalnodos,
            "Duración (ms)": tiempo,
            "Camino": resultado,
        }
    )

df = pd.DataFrame(resultados)
df

# ver el camino
# indice=0
# for i in range(len(df["Camino"][indice])):
#     clear_output(wait=True)
#     print(df["Camino"][indice][i])):
#     time.sleep(1)

```

# Actividades 
## Problema de la ruta entre ciudades utilizando algoritmos de búsqueda

Implementar:
    búsqueda en anchura (2 puntos)
    búsqueda en profundidad (2 puntos)
    búsqueda bidireccional (2 puntos)
    A* (4 puntos)

Utilizando las celdas necesarias, implementa los algoritmos de búsqueda para resolver el problema de la ruta entre dos ciudades en Rep. Dom. El código debe funcionar correctamente sin importar el orden de las dos ciudades y cuál ciudad se elija.

Detalles:

Debes implementar la clase "Nodo", que contendrá un estado y una referencia a su nodo padre, así como una manera de obtener la heurística en referencia a otro nodo.

Debe existir un mínimo de 15 ciudades, y cada ciudad debe tener una forma de conectarse con sus ciudades vecinas y generarlas como sucesores.

Debes implementar una clase "Ciudad" que contenga el nombre de la ciudad y una forma de generar sus sucesores que coincida con la implementación del nodo, además de una forma de calcular la distancia en comparación con otras ciudades.

Realiza pruebas para verificar que el código resuelve el problema correctamente. Es decir, puede dar rutas válidas con todos los algoritmos sin importar la combinación origen-destino que se elija para probar. Para realizar las pruebas, crea una celda que realice la búsqueda de rutas en todas las combinaciones posibles y almacene el resultado en un objeto DataFrame de pandas para poder comparar resultados.

